**Imports**

In [10]:
import os
import math
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from tqdm.notebook import tqdm

**Configuration & Paths**

In [11]:
CHUNK_DURATION = 5  # seconds
TARGET_SR = 16000   # 16kHz for Wav2Vec2 compatibility
INPUT_AUDIO_DIR = "../../data/processed/audio"
OUTPUT_AUDIO_DIR = "../../data/processed/audio_chunks"
METADATA_PATH = "../../data/processed/multimodal_dataset.csv"
OUTPUT_CSV_PATH = "../../data/processed/sequential_metadata.csv"

os.makedirs(OUTPUT_AUDIO_DIR, exist_ok=True)

**Load the Original Metadata**

In [12]:
print("Loading original metadata...")
df_meta = pd.read_csv(METADATA_PATH)

print(f"Total original files: {len(df_meta)}")
df_meta.head(3)

Loading original metadata...
Total original files: 800


,text,label,audio_path
0,"[Greetings], this is [Name]. I finally got my ...",0,../data/processed/audio/sample_0.wav
1,"[Greetings], this is [Name]. I am hosting a sm...",0,../data/processed/audio/sample_1.wav
2,"[Greetings], this is the [Company] contacting...",1,../data/processed/audio/sample_2.wav


**Slicing Engine**

In [ ]:
sequential_data = []
print(f"Slicing audio into {CHUNK_DURATION}-second chunks (with zero-padding)...")

for index, row in tqdm(df_meta.iterrows(), total=len(df_meta)):
    call_id = f"sample_{index}"
    audio_path = os.path.join(INPUT_AUDIO_DIR, f"{call_id}.wav")
    
    if not os.path.exists(audio_path):
        continue
        
    try:
        y, sr = librosa.load(audio_path, sr=TARGET_SR)
    except Exception as e:
        print(f"Error loading {audio_path}: {e}")
        continue

    samples_per_chunk = CHUNK_DURATION * TARGET_SR
    total_samples = len(y)
    
    # Use math.ceil to round up and include the final partial chunk
    num_chunks = math.ceil(total_samples / samples_per_chunk)

    for chunk_idx in range(num_chunks):
        start_sample = chunk_idx * samples_per_chunk
        end_sample = start_sample + samples_per_chunk
        chunk_audio = y[start_sample:end_sample]
        
        # Pad with zeros (silence) at the end of the array
        if len(chunk_audio) < samples_per_chunk:
            pad_length = samples_per_chunk - len(chunk_audio)
            chunk_audio = np.pad(chunk_audio, (0, pad_length), mode='constant')
        
        chunk_filename = f"{call_id}_chunk_{chunk_idx}.wav"
        chunk_filepath = os.path.join(OUTPUT_AUDIO_DIR, chunk_filename)
        sf.write(chunk_filepath, chunk_audio, TARGET_SR)
        
        sequential_data.append({
            "original_call_id": call_id,
            "chunk_index": chunk_idx,
            "chunk_path": f"../../data/processed/audio_chunks/{chunk_filename}",
            "label": row["label"]
        })

print("\n--- Slicing Complete ---")

Slicing audio into 5-second chunks (with zero-padding)...


  0%|          | 0/800 [00:00<?, ?it/s]


--- Slicing Complete ---


**Save the Sequential Dataset**

In [14]:
df_sequential = pd.DataFrame(sequential_data)
df_sequential.to_csv(OUTPUT_CSV_PATH, index=False)

print(f"Total 5-second chunks generated: {len(df_sequential)}")
print(f"Sequential metadata saved to: {OUTPUT_CSV_PATH}")
df_sequential.head()

Total 5-second chunks generated: 3958
Sequential metadata saved to: ../../data/processed/sequential_metadata.csv


,original_call_id,chunk_index,chunk_path,label
0,sample_0,0,../../data/processed/audio_chunks/sample_0_chu...,0
1,sample_0,1,../../data/processed/audio_chunks/sample_0_chu...,0
2,sample_0,2,../../data/processed/audio_chunks/sample_0_chu...,0
3,sample_1,0,../../data/processed/audio_chunks/sample_1_chu...,0
4,sample_1,1,../../data/processed/audio_chunks/sample_1_chu...,0
